# 🧪 Assignment 1 - Color-Based Object Detection and Analysis

## Objective and steps

In this assignment, you will identify and analyze objects in the image `shapes&color.png` based on their color. You have to apply the techniques learned during **Lab 1**. The colors to segment are: red, blue, yellow and green.

![image](\Images4notebook\shapes&colors.png)

Complete the provided notebook. Each section contains:
- Questions that you must answer to summarize your findings
- Key ideas that will help guide your implementation and analysis
- If you need more help, for each section there is a solution outline

### 📌 Step 1 – Color and Filter Comparison

1. The **original image**
2. The filtered image(s) obtained using the filters:
    - Averaging
    - Gaussian
3. Compare how filtering affects color detection performance

**📤 Output 1**

Produce one figure showing the comparison between:
- Color masks computed on the **orginal image**
- Color masks computed on the **filtered image(s)**

### 📌 Step 2 – Bounding Box Detection

Using the color masks chosen at the previous step:

1. Detect the objects corresponding to each color
2. Draw a box around each detected object

**📤 Output 2**

Produce one figure with 4 subplots, showing the boxes over the images. One subplot for each color.

### 📌 Step 3 – Object Area Extraction

For each detected object:

1. Extract the actual object area (not just the bounding box)
2. Display only the pixels belonging to the detected objects of that color

**📤 Output 3**

Produce one figure with 4 subplots, where each subplot shows only the detected objects for each color.

## 🛠️ Requirements and environment setup

Import all the necessary libraries and paths

In [ ]:
# Imports and paths

## 📌 Step 1 – Color and Filter Comparison

- Compute color masks on the original image
- Apply Filtering 
- Display Comparison

### 📊 Compute Hystograms for original image

**Histograms** help you understand pixel intensity distributions in each channel. 

This is useful for: 
1. Choosing threshold values for segmentation understanding 
2. How filtering changes intensities

In [ ]:
# Histograms

The global RGB histograms are dominated by a strong peak.
- Why does this happen?
- How could you modify the histogram computation so that the distribution of the colored objects becomes more visible?
 
<details>
<summary>🔎 🧠 Need a hint? Click here</summary>

Think about how white pixels are represented in RGB space and how you could build a mask to ignore them before computing the histogram.

</details>

In [ ]:
# More informative histograms

Based on the histogram analysis, determine the threshold conditions needed to create binary masks for each color (red, green, blue, yellow).

In [ ]:
# --- Example thresholds (TUNE THESE!) ---

# Red objects

# Blue objects

# Yellow objects

# Green objects


# Convert to uint8 masks


# Plot


<details>
<summary>👀 Need help? Solution Outline</summary>

If the RGB histograms show a very large peak near intensity 255 in all three channels, 
this strongly suggests that the white background dominates the image.

Since histograms represent global intensity distributions, a large amount of white pixels 
compresses the useful color information of the objects and makes threshold selection unreliable.

To obtain a more informative histogram:

1. Split the image into its three channels:
   - `b, g, r = cv2.split(img)`

2. Define a mask that excludes white (or nearly white) pixels.
   - A white pixel has high intensity in all three channels.
   - Keep pixels where at least one channel is significantly below 255.
   - Example logical rule:
     - `(r < T) OR (g < T) OR (b < T)`
   - Choose a practical threshold (e.g., T ≈ 230–245).

3. Convert the boolean mask to `uint8`, since OpenCV histogram functions require:
   - 0 for background
   - 255 for foreground

4. Compute RGB histograms using the mask parameter of `cv2.calcHist`.

This approach removes the dominant white background from the statistics 
and reveals the true intensity distribution of the colored objects, 
making segmentation threshold selection more meaningful.
</details>


###  🏻 Apply Filtering and Recompute color masks 🎭

Filtering can reduce noise and smooth intensity variations. This may improve segmentation, but it can also blur edges and reduce contrast.

We implement 3 different filters:

In [ ]:
# Define Filters

In [ ]:
# Recompute histograms and masks for each case

- Which is the best solution?
- Why?

<details>
<summary>👀 Need help? Solution Outline</summary>

To avoid code repetition and ensure a structured implementation:

1. Create a dictionary that stores all image variants (original and filtered versions), 
   using descriptive keys (e.g., "Original", "Box 3x3", "Gaussian").

2. Implement a function that computes the RGB histograms of a generic input image.
   The function should:
   - split the image into its three channels,
   - compute one histogram per channel,
   - return the three histograms.

3. Implement a second function that applies the same RGB threshold rules to a generic input image.
   The function should:
   - split the image into channels,
   - apply the threshold conditions,
   - return the binary masks for each color in a structured format.

Once these components are defined, iterate over the image variants and store 
the histograms and masks for each case.
</details>

**here you should have reached Output 1**

## 📌 Step 2 – Bounding Box Detection

Using the color masks chosen at the previous step:

1. Use morphological operations to improve the quality of the chosen masks
1. Detect the objects corresponding to each color
2. Draw a box around each detected object

### 💅 Refine Using Morphological Operations

Morphological operations help regularize the mask structure before proceeding to connected component analysis and bounding box extraction.

In [ ]:
# Refine the chosen masks. Plot.

### 📦 Bounding Box detection and drawing

Now, we can detect individual objects and draw a bounding box around each one. 

Given a binary mask where foreground pixels represent objects:

- How can you automatically separate different objects belonging to the same color?
- Once separated, how can you obtain the spatial extent (position and size) of each object?

Think in terms of regions of connected pixels rather than individual points.

To solve this task, you must use:

`cv2.connectedComponentsWithStats`

This function allows you to:

- Label each connected region in a binary image.
- Retrieve useful statistics for each region (position, size, area).
- Filter objects based on geometric properties.

Refer to the official documentation: [`cv2.connectedComponentsWithStats`](https://docs.opencv.org/3.4/d3/dc0/group__imgproc__shape.html#ga107a78bf7cd25dec05fb4dfc5c9e765f)

In [ ]:
# DRAW BBOX

<details>
<summary>👀 Need help? Solution Outline</summary>

A possible implementation strategy for your function (e.g., `draw_bboxes_from_mask`) is the following:

1. Convert the input mask into a binary image (values 0 and 1) to ensure proper connected component analysis.

2. Use a connected components algorithm (e.g., `cv2.connectedComponentsWithStats`) to:
   - identify each object,
   - extract bounding box coordinates,
   - retrieve area statistics.

3. Apply a minimum area threshold to discard very small components and reduce noise.

4. For each valid component:
   - extract the bounding box coordinates,
   - draw the corresponding rectangle on a copy of the original image.

The function should return the image with the bounding boxes drawn on it.
</details>

**here you should have reached Output 2**

## 📌 Step 3 – Object Area Extraction

1. Extract the actual object area (not just the bounding box)
2. Display only the pixels belonging to the detected objects of that color

### 📐 Compute area from masks

At this stage we want to extract the actual pixels belonging to each detected object (not the bounding box region). The simplest approach would be to apply the refined mask directly to the original image. 

Implement this basic solution first.

Use the function `cv2.findContours` to get binary object's contours.

Refer to the official documentation: [`cv2.findContours`](https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html#gadf1ad6a0b82947fa1fe3c3d497f260e0)

In [ ]:
# Implement basic solution - apply the refined mask

Sometimes, masks can contain small holes or fragmented regions, and edge-based contours can be broken for some objects. How can we have nicer contours?

In [ ]:
# Implement nicer contours solution

<details>
<summary>🔎 🧠 Need a hint? Click here</summary>

To obtain a more geometrically consistent contour, consider using a **gradient-based operator**.

</details>


<details>
<summary>👀 Need help? Solution Outline</summary>

A correct implementation should follow a structured pipeline for each color mask.

1. Convert the refined mask to a binary format (0/1) so that connected component analysis works correctly.

2. Use `cv2.connectedComponentsWithStats` to:
   - separate each object of the same color,
   - retrieve bounding box coordinates,
   - obtain the area of each component.

3. Remove noise components by discarding objects with area smaller than a chosen `min_area`.

4. For each valid component:
   - extract a Region Of Interest (ROI) using the bounding box,
   - add a small padding to avoid cutting object borders.

5. Inside the ROI:
   - build a stable filled object mask from the seed region (mask pixels),
   - optionally apply morphological closing to remove holes,
   - extract the largest external contour and fill it to obtain a clean object mask.

6. Create the cutout image:
   - start from a black image,
   - copy original image pixels only where the filled object mask is foreground.

7. Compute an edge-based contour for visualization:
   - apply Sobel gradients (x and y),
   - compute gradient magnitude,
   - threshold the magnitude (Otsu is acceptable),
   - apply morphological closing to connect broken edges,
   - extract contours and keep the largest one,
   - if the edge contour is unreliable, fall back to the seed contour.

8. Draw the selected contour on a global black image:
   - shift ROI contour coordinates to global image coordinates,
   - draw in white.

Your final outputs (per color) must include:
- a cutout image containing only object pixels,
- a contour image showing object boundaries.

The goal is robustness: objects must remain complete, contours must be stable, and small noisy components must not appear.
</details>


- Which is the best solution?
- Why?
- Show a comparison for the two results

**here you should have reached Output 3**